# Run streamlit app from a Google Colab Notebook
> Created by [Manuel Romero](https://twitter.com/mrm8488)

In [ ]:
!pip install -q streamlit

     |████████████████████████████████| 7.1MB 2.8MB/s 
     |████████████████████████████████| 4.4MB 45.5MB/s 
     |████████████████████████████████| 522kB 50.5MB/s 
     |████████████████████████████████| 112kB 51.3MB/s 
     |████████████████████████████████| 102kB 10.5MB/s 
     |████████████████████████████████| 122kB 48.4MB/s 
ERROR: google-colab 1.0.0 has requirement ipykernel~=4.10, but you'll have ipykernel 5.3.0 which is incompatible.
ERROR: google-auth 1.7.2 has requirement cachetools<3.2,>=2.0.0, but you'll have cachetools 4.1.0 which is incompatible.


## Create a streamlit app example


In [ ]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title="Gestion des Alertes & Chat SQL", layout="wide")

# Initialisation des données en session avec la clé "mwo" pour tout le monde
if "alerts" not in st.session_state:
    st.session_state.alerts = [
        {"id": 1, "name": "Stock Critique", "query": "SELECT * FROM stock < 5", "alerts_30d": 12, "desc": "Vérifie les stocks bas.", "mwo": True},
        {"id": 2, "name": "Erreurs 500", "query": "SELECT count(*) FROM logs...", "alerts_30d": 3, "desc": "Pic d'erreurs serveur.", "mwo": False}
    ]

# Historique du chat et dernier output généré
if "chat_messages" not in st.session_state:
    st.session_state.chat_messages = [
        {"role": "assistant", "content": "Bonjour ! Décrivez l'alerte que vous souhaitez créer et je générerai la requête SQL."}
    ]
if "generated_query" not in st.session_state:
    st.session_state.generated_query = ""

# Layout principal : 3 colonnes [Chat | Formulaire de création (Carte) | Alertes existantes]
col_left, col_right = st.columns([1, 2], gap="medium")

# ==========================================
# COLONNE 1 : LE CHAT (Génération SQL)
# ==========================================
with col_left:
    st.subheader("Alert Creation")
    with st.container(border=True, height=100):
        for msg in st.session_state.chat_messages:
            with st.chat_message(msg["role"]):
                st.write(msg["content"])

    user_prompt = st.chat_input("Ex: Trouve les utilisateurs inactifs depuis 30 jours")
    if user_prompt:
        st.session_state.chat_messages.append({"role": "user", "content": user_prompt})

        simulated_sql = f"SELECT * FROM users WHERE status = 'inactive' -- généré pour: {user_prompt}"
        st.session_state.generated_query = simulated_sql

        assistant_reply = f"Voici la requête SQL générée :\n```sql\n{simulated_sql}\n```"
        st.session_state.chat_messages.append({"role": "assistant", "content": assistant_reply})
        st.rerun()

    if st.session_state.generated_query:
        if st.button("📥 Créer une carte depuis le Chat", use_container_width=True, type="secondary"):
            st.success("Requête injectée dans le formulaire de gauche !")

    # ==========================================
    # COLONNE 2 : ENCART DE CRÉATION (Format Carte)
    # ==========================================
    with st.container(border=True):
        default_query = st.session_state.generated_query if st.session_state.generated_query else ""

        form_name = st.text(f"Nom de la formule - 312 Alertes (30 derniers jours)")
        sql_content = st.text_area("Contenu requête SQL", value=default_query, placeholder="SELECT ...", height=70, label_visibility="collapsed")
        description = st.text_area("Description (modifiable)", placeholder="Description...", height=70)

        c_btn1, c_btn2 = st.columns(2)
        with c_btn1:
            add_btn = st.button("Add", use_container_width=True, type="primary")
        with c_btn2:
            mwo_init_btn = st.button("Activate MWO", use_container_width=True)

        if add_btn:
            if form_name and sql_content:
                new_id = max([a["id"] for a in st.session_state.alerts], default=0) + 1
                st.session_state.alerts.append({
                    "id": new_id,
                    "name": form_name,
                    "query": sql_content,
                    "alerts_30d": alerts_count,
                    "desc": description,
                    "mwo": mwo_init_btn
                })
                st.session_state.generated_query = ""
                st.rerun()
            else:
                st.warning("Nom et SQL requis.")

# ==========================================
# COLONNE 3 : CARTES DES ALERTES EXISTANTES
# ==========================================
with col_right:
    st.subheader("📊 Alertes existantes")

    if not st.session_state.alerts:
        st.info("Aucune alerte enregistrée.")
    else:
        for alert in st.session_state.alerts:
            with st.container(border=True):
                c1, c2, b1, b2 = st.columns([3, 1, 1, 1])
                with c1:
                    status = "🟢 MWO ON" if alert.get("mwo", False) else "⚪ MWO OFF"
                    st.markdown(f"**{alert['name']}** &nbsp; `{status}`")
                with c2:
                    st.caption(f"30j: **{alert['alerts_30d']} alerts**")

                with b1:
                    if st.button("🗑️ Remove", key=f"rm_{alert['id']}", use_container_width=True):
                        st.session_state.alerts = [a for a in st.session_state.alerts if a["id"] != alert["id"]]
                        st.rerun()
                with b2:
                    mwo_label = "Désactiver MWO" if alert.get("mwo", False) else "Activate MWO"
                    if st.button(mwo_label, key=f"mwo_{alert['id']}", use_container_width=True):
                        alert["mwo"] = not alert.get("mwo", False)
                        st.rerun()

Writing app.py


## Install localtunnel

In [ ]:
!npm install localtunnel

npm WARN saveError ENOENT: no such file or directory, open '/content/package.json'
npm notice created a lockfile as package-lock.json. You should commit this file.
npm WARN enoent ENOENT: no such file or directory, open '/content/package.json'
npm WARN content No description
npm WARN content No repository field.
npm WARN content No README data
npm WARN content No license field.

+ localtunnel@2.0.2
added 22 packages from 22 contributors and audited 22 packages in 2.261s

3 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities



## Run streamlit in background

In [ ]:
!streamlit run /content/app.py &>/content/logs.txt &

## Expose the port 8501
Then just click in the `url` showed.

A `log.txt`file will be created.

In [ ]:
!npx localtunnel --port 8501

npx: installed 22 in 1.848s
your url is: https://shaggy-eagles-dig-34-125-103-60.loca.lt


[![ko-fi](https://www.ko-fi.com/img/githubbutton_sm.svg)](https://ko-fi.com/Y8Y3VYYE)